# Codefest 3 CLLM — CUDA GEMM: Naive vs. Tiled
**ECE 410/510 Spring 2026**

Before running: Runtime → Change runtime type → **T4 GPU**

In [ ]:
# Confirm GPU
!nvidia-smi
!nvcc --version

In [ ]:
# Upload gemm_naive.cu and gemm_tiled.cu using the file panel on the left,
# OR paste them below using %%writefile.
# Then compile both kernels:
!nvcc -O2 -o gemm_naive gemm_naive.cu
!nvcc -O2 -o gemm_tiled gemm_tiled.cu

In [ ]:
# Run both kernels
print('--- Naive ---')
!./gemm_naive
print()
print('--- Tiled ---')
!./gemm_tiled

In [ ]:
# Nsight Compute profiling (may need --set full on some Colab instances)
!ncu --metrics sm__throughput.avg.pct_of_peak_sustained_elapsed,\
dram__throughput.avg.pct_of_peak_sustained_elapsed,\
l1tex__t_bytes.sum,dram__bytes.sum \
./gemm_naive 2>&1 | tail -30

In [ ]:
!ncu --metrics sm__throughput.avg.pct_of_peak_sustained_elapsed,\
dram__throughput.avg.pct_of_peak_sustained_elapsed,\
l1tex__t_bytes.sum,dram__bytes.sum \
./gemm_tiled 2>&1 | tail -30

In [ ]:
# ── Roofline plot ─────────────────────────────────────────────────────────
# Fill in the measured GFLOP/s values from the kernel output above.
# T4 GPU specs: peak FP32 = 8,100 GFLOPS, peak BW = 300 GB/s

import numpy as np
import matplotlib.pyplot as plt

# ── Hardware ──────────────────────────────────────────────────────────────
peak_compute = 8100   # GFLOPS (T4 FP32)
peak_bw      = 300    # GB/s
ridge        = peak_compute / peak_bw   # 27 FLOP/byte

# ── Kernel stats (fill in from your run output) ───────────────────────────
N = 1024
flops    = 2.0 * N**3
bytes_rw = 3.0 * N**2 * 4
ai       = flops / bytes_rw   # 170.67 FLOP/byte (same for both kernels)

gflops_naive = 0.0   # <-- FILL IN from ./gemm_naive output
gflops_tiled = 0.0   # <-- FILL IN from ./gemm_tiled output

# ── Roofline curve ────────────────────────────────────────────────────────
ai_x     = np.logspace(-1, 4, 2000)
roofline = np.minimum(ai_x * peak_bw, peak_compute)

fig, ax = plt.subplots(figsize=(9, 6))
ax.loglog(ai_x, roofline, 'k-', linewidth=2, label='T4 Roofline')
ax.axvline(ridge, color='gray', linestyle='--', linewidth=1)
ax.text(ridge * 1.1, 20, f'Ridge\n{ridge:.0f} F/B', fontsize=8, color='gray')

ax.plot(ai, gflops_naive, 'o', color='tomato',    markersize=10, zorder=5,
        label=f'Naive GEMM  — {gflops_naive:.0f} GFLOP/s')
ax.plot(ai, gflops_tiled, 's', color='steelblue', markersize=10, zorder=5,
        label=f'Tiled GEMM  — {gflops_tiled:.0f} GFLOP/s')

ax.annotate(f'Naive\n{gflops_naive:.0f} GFLOP/s',
            xy=(ai, gflops_naive), xytext=(ai*0.15, gflops_naive*0.4),
            fontsize=8, color='tomato',
            arrowprops=dict(arrowstyle='->', color='tomato'))
ax.annotate(f'Tiled\n{gflops_tiled:.0f} GFLOP/s',
            xy=(ai, gflops_tiled), xytext=(ai*0.15, gflops_tiled*2.5),
            fontsize=8, color='steelblue',
            arrowprops=dict(arrowstyle='->', color='steelblue'))

ax.set_xlabel('Arithmetic Intensity (FLOP/byte)', fontsize=11)
ax.set_ylabel('Attainable Performance (GFLOPS)',   fontsize=11)
ax.set_title('CUDA GEMM Roofline — NVIDIA T4 GPU\nNaive vs. Tiled (tile=8), N=1024 FP32',
             fontsize=11, fontweight='bold')
ax.set_xlim(1e-1, 1e4)
ax.set_ylim(1e0,  1e5)
ax.grid(True, which='both', linestyle=':', alpha=0.5)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('gemm_roofline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved gemm_roofline.png — download and add to codefest/cf03/profiling/')